In [2]:
import os
import pandas as pd

In [ ]:
#This script filters the data generated by the analysis pipeline, where the nucleus of each cell in the field is thresholded and a number of parameters for each cell extracted, with the final output a csv file.
#Filters applied to exclude mitotic cells, cells with very large or small nuclei, and cells with very low or very high intensity for the channel of interest.

# Specify your input and output directories
input_dir = "your_input_folder/"
output_dir = "your_output_folder/output_filtered/"

# Process each CSV file in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        # Read the CSV file
        df = pd.read_csv(file_path)
             
        # Set thresholds based on standardised parameters to account for differences between cell lines and replicates
        threshold_dna_max = (df.mean_hoechst.mean()) * 1.6
        threshold_dna_min = (df.mean_hoechst.mean()) / 1.75
        threshold_size_max = (df.nuclear_area_microns.mean()) * 2.1

        # Example filtering parameters
        filtered_df = df[            
            (df['nuclear_area_microns'] > 16) &
            (df['nuclear_area_microns'] < threshold_size_max) &
            (df['mean_hoechst'] < threshold_dna_max) & (df['mean_hoechst'] > threshold_dna_min) &
            (df['mean_hoechst'] > 6500) &
            (df['mean_kat2b'] > 1000) & (df['mean_kat2b'] < 40000) &
            (df['mean_h3s10p'] > 750) & (df['mean_h3s10p'] < 930)
        ]
            
        # Specify the new column order
        new_column_order = [
            'image_name', 'nuclear_area_microns', 'mean_hoechst', 'mean_kat2b', 'mean_h3s10p', 'condition', 'replicate', 'identifier']
        
        # Reorder the columns
        filtered_df = filtered_df.reindex(columns=new_column_order)

        # Save the filtered DataFrame to a new CSV file in the output directory
        filtered_df.to_csv(os.path.join(output_dir, filename), index=False)
